# Bronze — table list

First task of the Bronze stage. It does two things:

1. syncs `config/ingestion_config.yaml` into `control.tables` (config-as-code stays the
   source of truth; the control table is what runtime reads)
2. emits the enabled table names as a **task value**, which the downstream `for_each`
   task fans out over

Keeping the list here rather than hardcoded in the Job definition is what keeps the
pipeline metadata-driven: adding a table is a YAML edit, not a Job edit.

In [ ]:
import json
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
SRC = os.path.join(REPO_ROOT, "src")

if SRC not in sys.path:
    sys.path.insert(0, SRC)

In [ ]:
from common.config import load_configs, sync_config_to_control, get_enabled_table_configs

CONFIG_PATH = os.path.join(REPO_ROOT, "config", "ingestion_config.yaml")

# YAML -> control.tables. Done ONCE here rather than in every for_each instance.
synced = sync_config_to_control(spark, load_configs(CONFIG_PATH))
print(f"synced {synced} table configs to control.tables")

tables = [cfg["source_table"] for cfg in get_enabled_table_configs(spark)]
print(f"{len(tables)} enabled tables:")
for name in tables:
    print(" ", name)

## Emit the list for the `for_each` task

The downstream task references this as `{{tasks.bronze_list.values.tables}}`.

In [ ]:
dbutils.jobs.taskValues.set(key="tables", value=tables)

# Also emit as JSON for eyeballing in the run output
print(json.dumps(tables, indent=2))